# 02 · Joins & Window Functions

Multi-table joins and window functions on enriched sales.

`sales` here is already typed and cleaned for you via `read_sales_typed` (you
built that by hand in notebook 03's territory). The store/category text is
normalized in setup so grouping behaves — text cleaning is covered in
notebook 03.

In [ ]:
import sys
from pathlib import Path

_root = Path.cwd()
while not (_root / "utils" / "spark.py").exists() and _root != _root.parent:
    _root = _root.parent
for _p in (str(_root), str(_root / "src")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from utils.spark import build_session, read_dim, read_sales_raw, read_sales_typed

spark = build_session("nb02")
print("Spark", spark.version, "ready")

sales = read_sales_typed(spark)
# Normalize dimension text up front so groupBy keys collapse cleanly.
# (The casing/whitespace cleaning technique itself is taught in notebook 03.)
products = read_dim(spark, "dim_products").withColumn("category", F.initcap(F.trim("category")))
stores = read_dim(spark, "dim_stores").withColumn("region", F.initcap(F.trim("region")))
customers = read_dim(spark, "dim_customers")
print("typed sales rows:", sales.count())

## Task 1: Enrich sales with dimensions

Inner-join `sales` to `products` (on `product_id`) and `stores` (on `store_id`) to add `category`, `brand`, `region`, `channel`. Name the result `enriched`.

Note: some sales reference non-existent products (orphan FKs); an inner join drops them, which is fine here.

In [ ]:
# TODO
enriched = sales

In [ ]:
# ✅ CHECK — run this cell to grade your answer
for _c in ["category", "brand", "region", "channel"]:
    assert _c in enriched.columns, _c
# An inner join drops orphan FKs but must NOT multiply rows (dim keys are unique).
assert enriched.count() == enriched.select("txn_id").distinct().count(), "join fanned out — dim key not unique?"
assert enriched.count() <= sales.count()                        # orphans dropped, never added
assert enriched.count() > 0.95 * sales.count(), "too many rows lost — wrong join key or join type?"
# The attached columns must really be sourced from the dimensions, not garbage/wrong columns.
_pbrands = {r["brand"] for r in products.select("brand").distinct().collect()}
_ebrands = {r["brand"] for r in enriched.select("brand").distinct().collect()}
assert _ebrands <= _pbrands and len(_ebrands) > 1, "brand not sourced from dim_products"
print("✅ Task 1 passed — enriched rows:", enriched.count())

## Task 2: Top 3 brands per region

Using `enriched`, total revenue per `(region, brand)`, then rank brands within each region by revenue and keep the top 3. Name it `top_brands` with columns `region`, `brand`, `brand_revenue`, `rnk`. Use a window + `row_number`.

In [ ]:
# TODO
top_brands = enriched

In [ ]:
# ✅ CHECK — run this cell to grade your answer
from collections import defaultdict
assert {"region", "brand", "brand_revenue", "rnk"} <= set(top_brands.columns)

# Independent ground truth: top-3 brands by revenue within EACH region.
_truth = enriched.groupBy("region", "brand").agg(F.sum("revenue").alias("rev"))
_regions = [r["region"] for r in _truth.select("region").distinct().collect()]
_exp = {}
for _reg in _regions:
    _rows = _truth.filter(F.col("region") == _reg).orderBy(F.col("rev").desc()).limit(3).collect()
    _exp[_reg] = {r["brand"]: round(r["rev"], 2) for r in _rows}

_got, _rank_rev = defaultdict(dict), defaultdict(dict)
for r in top_brands.collect():
    _got[r["region"]][r["brand"]] = round(r["brand_revenue"], 2)
    _rank_rev[r["region"]][int(r["rnk"])] = r["brand_revenue"]

# Every region must be represented — catches a global LIMIT that keeps only one region.
assert set(_got) == set(_regions), ("missing regions:", set(_regions) - set(_got))
for _reg in _regions:
    assert set(_got[_reg]) == set(_exp[_reg]), (_reg, "wrong brands", _got[_reg], _exp[_reg])
    for _b, _v in _exp[_reg].items():
        assert abs(_got[_reg][_b] - _v) < 1.0, (_reg, _b, "revenue off")
    # ranks are exactly 1,2,3 and ordered by descending revenue
    assert set(_rank_rev[_reg]) == {1, 2, 3}, (_reg, "ranks", set(_rank_rev[_reg]))
    assert _rank_rev[_reg][1] >= _rank_rev[_reg][2] >= _rank_rev[_reg][3], (_reg, "rank order")
print("✅ Task 2 passed —", len(_regions), "regions × top-3 brands")

## Task 3: Broadcast join

Re-do the product join but explicitly **broadcast** the small `products` dimension to avoid a shuffle. Name it `enriched_bc`. (Use `F.broadcast`.)

In [ ]:
# TODO
enriched_bc = sales

In [ ]:
# ✅ CHECK — run this cell to grade your answer
_plan = enriched_bc._jdf.queryExecution().executedPlan().toString()
assert "Broadcast" in _plan, "expected a broadcast join in the physical plan"
# Broadcasting is an optimisation only — the result must match a plain product join.
assert {"category", "brand"} <= set(enriched_bc.columns)
_ref = sales.join(products, on="product_id", how="inner")
assert enriched_bc.count() == _ref.count(), (enriched_bc.count(), _ref.count())
print("✅ Task 3 passed — broadcast join confirmed")

## Task 4: Rolling 7-day revenue

Aggregate `enriched` to daily total revenue (`daily_revenue`), then add a 7-day trailing sum `rolling_7d` using a window ordered by date with `rowsBetween(-6, 0)`. Name the result `daily` with columns `txn_date`, `daily_revenue`, `rolling_7d`, ordered by date.

In [ ]:
# TODO
daily = enriched

In [ ]:
# ✅ CHECK — run this cell to grade your answer
assert {"txn_date", "daily_revenue", "rolling_7d"} <= set(daily.columns)
_rows = daily.orderBy("txn_date").collect()
# One row per day, ascending by date.
_dates = [r["txn_date"] for r in _rows]
assert _dates == sorted(_dates) and len(_dates) == len(set(_dates)), "daily must be one row per date, ordered"
# First day's trailing-7 sum is just its own revenue.
assert abs(_rows[0]["rolling_7d"] - _rows[0]["daily_revenue"]) < 1e-6, "row 0 rolling != its own day"
# At row i, rolling_7d must equal the sum of daily_revenue over the trailing window [i-6, i].
for _i in (6, len(_rows) // 2, len(_rows) - 1):
    _exp = sum(r["daily_revenue"] for r in _rows[max(0, _i - 6):_i + 1])
    assert abs(_rows[_i]["rolling_7d"] - _exp) < 1.0, (_i, _rows[_i]["rolling_7d"], _exp)
print("✅ Task 4 passed —", len(_rows), "days")

## Task 5: Week-over-week growth (lag)

From `enriched`, compute weekly revenue. Build a `year_week` key as `concat_ws("-W", year(txn_date), lpad(weekofyear(txn_date), 2, "0"))` (e.g. `2026-W23`). Then use `lag` to get the previous week's revenue and a `wow_pct` growth percentage. Name it `wow` with columns `year_week`, `weekly_revenue`, `prev_week`, `wow_pct`, ordered by week.

In [ ]:
# TODO
wow = enriched

In [ ]:
# ✅ CHECK — run this cell to grade your answer
assert {"year_week", "weekly_revenue", "prev_week", "wow_pct"} <= set(wow.columns)
_rows = wow.orderBy("year_week").collect()
assert len(_rows) == len({r["year_week"] for r in _rows}), "one row per year_week"
assert _rows[0]["prev_week"] is None, "first week has no previous"
for _i in range(1, len(_rows)):
    # prev_week must be the lag (previous week's) weekly_revenue
    assert abs(_rows[_i]["prev_week"] - _rows[_i - 1]["weekly_revenue"]) < 1e-6, (_i, "prev_week wrong")
    # wow_pct must equal (weekly - prev) / prev * 100
    _exp = (_rows[_i]["weekly_revenue"] - _rows[_i]["prev_week"]) / _rows[_i]["prev_week"] * 100
    assert abs(_rows[_i]["wow_pct"] - _exp) < 0.1, (_i, _rows[_i]["wow_pct"], _exp)
print("✅ Task 5 passed —", len(_rows), "weeks")

## Task 6: Customer RFM scoring

For sales with a non-null `customer_id`, compute per customer: `recency_days` (days from last purchase to 2026-06-05), `frequency` (#txns), `monetary` (total revenue). Then add `m_quartile` = `ntile(4)` over monetary. Name it `rfm` with columns `customer_id`, `recency_days`, `frequency`, `monetary`, `m_quartile`. (Use `sales`, which still has `customer_id`.)

In [ ]:
# TODO
rfm = sales

In [ ]:
# ✅ CHECK — run this cell to grade your answer
import datetime
assert {"customer_id", "recency_days", "frequency", "monetary", "m_quartile"} <= set(rfm.columns)
# One row per customer, and only non-null customers were scored.
assert rfm.filter(F.col("customer_id").isNull()).count() == 0, "null customer leaked in"
assert rfm.count() == rfm.select("customer_id").distinct().count(), "duplicate customers — group by customer_id"
# frequency must reconcile with the raw transaction counts.
_src = sales.filter(F.col("customer_id").isNotNull())
assert rfm.agg(F.sum("frequency")).first()[0] == _src.count(), "frequency != #txns per customer"
# m_quartile is ntile(4): exactly {1,2,3,4} and near-equal bucket sizes.
assert {r["m_quartile"] for r in rfm.select("m_quartile").distinct().collect()} == {1, 2, 3, 4}
_sizes = [r["c"] for r in rfm.groupBy("m_quartile").agg(F.count("*").alias("c")).collect()]
assert max(_sizes) - min(_sizes) <= 1, ("ntile buckets unbalanced", _sizes)
# recency_days = days from last purchase to 2026-06-05: non-negative and matches a recompute.
assert rfm.filter(F.col("recency_days") < 0).count() == 0, "negative recency"
_cust = rfm.orderBy(F.col("monetary").desc()).first()["customer_id"]
_last = _src.filter(F.col("customer_id") == _cust).agg(F.max("txn_date")).first()[0]
_exp_rec = (datetime.date(2026, 6, 5) - _last).days
assert rfm.filter(F.col("customer_id") == _cust).first()["recency_days"] == _exp_rec, "recency_days formula wrong"
print("✅ Task 6 passed —", rfm.count(), "customers")

---
🎉 Finished! Re-run the notebook top-to-bottom; every CHECK cell should print a ✅.